# Predict Customer Churn #
This is my first Kaggle competition entry, working on on season 6 episode 3 of the playground predictions series - Predict Customer Churn

In [2]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
import torch.nn as nn
import torch.nn.functional as F

Let's import our data as pandas dataframes for easy manipulation.

In [4]:
train_df = pd.read_csv("data/train.csv")
test_df = pd.read_csv("data/test.csv")

N_tr = len(train_df)
N_te = len(test_df)

print(f"Training Data Length: {N_tr}\nTest Data Length : {N_te}")

Training Data Length: 594194
Test Data Length : 254655


Now check that whether any data is missing, this may take a while...

In [19]:
invalid_entries = {column: 0 for column in train_df.columns}
for i in range(N_tr):
    for col in train_df.columns:
        invalid_entries[col] += pd.isnull(train_df.loc[i, col]) # Check train dataframe for null values.
        if i < N_te and col != "Churn":
            invalid_entries[col] += pd.isnull(test_df.loc[i, col]) # Check test dataframe as well.
print(invalid_entries)

{'id': 0, 'gender': 0, 'SeniorCitizen': 0, 'Partner': 0, 'Dependents': 0, 'tenure': 0, 'PhoneService': 0, 'MultipleLines': 0, 'InternetService': 0, 'OnlineSecurity': 0, 'OnlineBackup': 0, 'DeviceProtection': 0, 'TechSupport': 0, 'StreamingTV': 0, 'StreamingMovies': 0, 'Contract': 0, 'PaperlessBilling': 0, 'PaymentMethod': 0, 'MonthlyCharges': 0, 'TotalCharges': 0, 'Churn': 0}


No problems, a miracle! Now we need to clean the data up, categorical data is no good to our model, so we'll use a one-hot encoder to convert our categorical data to data we can use in our model. We use the pd.get_dummies method for this, which converts a categorical data column into several boolean data columns, with the downside that this makes the data significantly harder to read. We should also normalise our data between the range 0 and 1 to prevent larger raw values from having a disproportionate impact on our output.

In [5]:
def normalise(df: pd.DataFrame) -> pd.DataFrame:
    types = df.dtypes
    cols =[col for col in df.columns if types[col] != "str" and col != "id"]
    for col in cols:
        df_col = df[col]
        df[col] = (df_col-df_col.mean())/df_col.std()
    return df

In [6]:
def encode_one_hot(df: pd.DataFrame) -> pd.DataFrame:
    types = df.dtypes
    df = pd.get_dummies(df, columns=[col for col in df.columns if types[col] == "str"])
    return df

In [7]:
train_df = encode_one_hot(normalise(train_df))
test_df = encode_one_hot(normalise(test_df))

Now let's define a neural network using pytorch, with an MLP structure due to our data being tabulated. Our input size is 48, 1 for each column in our dataframe, then we have a hidden layer with 64 nodes, and finally an output layer with a single output, this is our probability of churn for our input.

In [ ]:
class mlp_V1(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.fc1 = nn.Linear(48, 64)
        self.fc2 = nn.Linear(64, 1)

    def forward(self, x):
        x = self.fc1(x)
        x = self.fc2(x)
        return x